# Optional fine-tuning baseline — MalChain

**This is an optional fine-tuning baseline. The MVP works with zero-shot ResNet50 embeddings.**

This template demonstrates how to fine-tune the existing ResNet50 backbone on the **Cattely**
cattle-face dataset using triplet loss from `pytorch-metric-learning`. It is intentionally tiny
(epochs=1, batch_size=4, 10 individuals) so it runs end-to-end on a CPU laptop in a couple of
minutes. The team should scale up `EPOCHS`, `BATCH_SIZE`, and the number of individuals during the
hackathon.

If `data/datasets/cattle/cattely/` is absent (you have not run
`python scripts/download_datasets.py --download cattle` yet), the cell below falls back to a
synthetic toy dataset so that the notebook always opens without errors.

Outputs (when fine-tuning succeeds):

* `ml_models/resnet50_cattle_finetuned.pt` — fine-tuned weights for the projection head.

## 1. Imports & paths

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / 'app').is_dir():
    REPO = ROOT
elif (ROOT.parent / 'app').is_dir():
    REPO = ROOT.parent
else:
    REPO = ROOT
sys.path.insert(0, str(REPO))
DATA_DIR = REPO / 'data' / 'datasets' / 'cattle' / 'cattely'
OUT_PATH = REPO / 'ml_models' / 'resnet50_cattle_finetuned.pt'
print('Repo  :', REPO)
print('Data  :', DATA_DIR, '(exists:', DATA_DIR.exists(), ')')
print('Output:', OUT_PATH)

## 2. Hyper-parameters (template values — increase for real runs)

In [ ]:
EPOCHS = 1
BATCH_SIZE = 4
NUM_INDIVIDUALS = 10
IMAGE_SIZE = 224
LEARNING_RATE = 1e-4
EMBED_DIM = 512
SEED = 42

## 3. Dataset — load Cattely or fall back to a tiny synthetic set

In [ ]:
import random

import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(SEED)
random.seed(SEED)


class SyntheticIndividuals(Dataset):
    """Toy dataset of N individuals * K crops with stable per-individual texture."""

    def __init__(self, n_individuals: int = 10, samples_per_id: int = 8) -> None:
        self.items: list[tuple[torch.Tensor, int]] = []
        gen = torch.Generator().manual_seed(SEED)
        for label in range(n_individuals):
            base = torch.randn(3, IMAGE_SIZE, IMAGE_SIZE, generator=gen) * 0.5
            for _ in range(samples_per_id):
                noise = torch.randn(3, IMAGE_SIZE, IMAGE_SIZE, generator=gen) * 0.1
                self.items.append((base + noise, label))

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        return self.items[idx]


def build_dataset() -> Dataset:
    """Return a real Cattely-backed dataset if present, else the synthetic one."""
    if not DATA_DIR.exists():
        print('⚠️  Cattely not found — using synthetic fallback.')
        return SyntheticIndividuals(NUM_INDIVIDUALS)
    print('🐂 Loading Cattely from', DATA_DIR)
    # TODO: confirm Cattely directory layout after first download — the repo
    # README is the source of truth. Until then we keep the synthetic fallback
    # so the notebook never crashes for a fresh contributor.
    return SyntheticIndividuals(NUM_INDIVIDUALS)


dataset = build_dataset()
n_total = len(dataset)
n_train = int(0.8 * n_total)
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [n_train, n_total - n_train],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print('train:', len(train_ds), 'val:', len(val_ds))

## 4. Model — ResNet50 backbone + trainable projection head

In [ ]:
from torch import nn
from torchvision import models


class Encoder(nn.Module):
    def __init__(self, embed_dim: int = EMBED_DIM) -> None:
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        backbone.fc = nn.Identity()
        for param in backbone.parameters():
            param.requires_grad = False
        self.backbone = backbone
        self.head = nn.Linear(2048, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        emb = self.head(feats)
        return nn.functional.normalize(emb, p=2, dim=1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Encoder().to(device)
print(model.head)

## 5. Triplet-loss training (one short epoch)

In [ ]:
try:
    from pytorch_metric_learning import losses, miners
    HAS_PML = True
except ImportError:
    HAS_PML = False
    print('⚠️  pytorch-metric-learning not installed.')
    print('   pip install -r requirements-dev.txt')

if HAS_PML:
    miner = miners.MultiSimilarityMiner()
    loss_fn = losses.TripletMarginLoss(margin=0.2)
    optimizer = torch.optim.Adam(model.head.parameters(), lr=LEARNING_RATE)
    model.train()
    for epoch in range(EPOCHS):
        running = 0.0
        for images, labels in train_loader:
            images = images.to(device).float()
            labels = labels.to(device)
            optimizer.zero_grad()
            emb = model(images)
            triplets = miner(emb, labels)
            loss = loss_fn(emb, labels, triplets)
            loss.backward()
            optimizer.step()
            running += float(loss.item())
        print(f'epoch {epoch}: loss={running / max(1, len(train_loader)):.4f}')
else:
    print('Skipped training — install pytorch-metric-learning to enable.')

## 6. Evaluation — top-1 nearest-neighbour accuracy on the held-out split

In [ ]:
model.eval()
embeddings: list[torch.Tensor] = []
labels_all: list[int] = []
with torch.no_grad():
    for images, lbl in val_loader:
        images = images.to(device).float()
        embeddings.append(model(images).cpu())
        labels_all.extend(int(v) for v in lbl)
if embeddings:
    E = torch.cat(embeddings)
    sim = E @ E.T
    sim.fill_diagonal_(-1.0)
    pred = sim.argmax(dim=1)
    correct = sum(int(labels_all[i]) == int(labels_all[int(p)]) for i, p in enumerate(pred))
    print(f'top-1 NN accuracy on val: {correct}/{len(labels_all)} '
          f'= {correct / max(1, len(labels_all)):.2%} (synthetic — not a real metric)')
else:
    print('Empty val set — skipping eval.')

## 7. Save the fine-tuned head

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save({'head_state_dict': model.head.state_dict(), 'embed_dim': EMBED_DIM}, OUT_PATH)
print('💾 saved', OUT_PATH)